In [3]:
import numpy as np
import pandas as pd
df_matches = pd.read_csv('../data/matches_clean.csv')
df_del     = pd.read_csv('../data/deliveries_clean.csv')
print(f"Matches    : {df_matches.shape}")
print(f"Deliveries : {df_del.shape}")
def get_phase(over):
    if over <= 6:
        return 'Powerplay'
    elif over <= 15:
        return 'Middle'
    else:
        return 'Death'
df_del['phase'] = df_del['over'].apply(get_phase)
print("Phase distribution:")
print(df_del['phase'].value_counts())
df_del['is_boundary'] = df_del['batsman_runs'].apply(
    lambda x: 1 if x in [4, 6] else 0
)
print(f"\nTotal boundaries: {df_del['is_boundary'].sum():,}")
df_del['is_dot_ball'] = (
    (df_del['total_runs'] == 0)
).astype(int)
print(f"Total dot balls: {df_del['is_dot_ball'].sum():,}")
df_del['is_wicket'] = df_del['player_dismissed'].apply(
    lambda x: 0 if x == 'not_out' else 1
)
print(f"Total wickets  : {df_del['is_wicket'].sum():,}")
batsman_match = df_del.groupby(['match_id', 'batter']).agg(
    runs_scored   = ('batsman_runs', 'sum'),
    balls_faced   = ('batsman_runs', 'count'),
    fours         = ('batsman_runs', lambda x: (x == 4).sum()),
    sixes         = ('batsman_runs', lambda x: (x == 6).sum()),
).reset_index()
batsman_match['strike_rate'] = (
    batsman_match['runs_scored'] / batsman_match['balls_faced'] * 100
).round(2)
print("\nBatsman match stats sample:")
print(batsman_match.head())
df_del['is_legal'] = df_del['extras_type'].apply(
    lambda x: 0 if str(x) in ['wides', 'noballs'] else 1
)
bowler_match = df_del.groupby(['match_id', 'bowler']).agg(
    runs_conceded  = ('total_runs', 'sum'),
    legal_balls    = ('is_legal', 'sum'),
    wickets_taken  = ('is_wicket', 'sum'),
    dot_balls      = ('is_dot_ball', 'sum'),
).reset_index()
bowler_match['overs_bowled']  = (bowler_match['legal_balls'] / 6).round(2)
bowler_match['economy_rate']  = (
    bowler_match['runs_conceded'] / bowler_match['overs_bowled']
).replace([np.inf, -np.inf], 0).round(2)
bowler_match['dot_ball_pct']  = (
    bowler_match['dot_balls'] / bowler_match['legal_balls'] * 100
).round(2)
print("\nBowler match stats sample:")
print(bowler_match.head())
phase_summary = df_del.groupby('phase').agg(
    total_runs    = ('total_runs', 'sum'),
    total_wickets = ('is_wicket', 'sum'),
    total_balls   = ('total_runs', 'count'),
    boundaries    = ('is_boundary', 'sum'),
).reset_index()
phase_summary['run_pct']    = (phase_summary['total_runs'] / phase_summary['total_runs'].sum() * 100).round(2)
phase_summary['wicket_pct'] = (phase_summary['total_wickets'] / phase_summary['total_wickets'].sum() * 100).round(2)
phase_summary['run_rate']   = (phase_summary['total_runs'] / (phase_summary['total_balls'] / 6)).round(2)
print("\nPhase-wise Summary:")
print(phase_summary)
df_del.to_csv('../data/deliveries_featured.csv', index=False)
batsman_match.to_csv('../data/batsman_match_stats.csv', index=False)
bowler_match.to_csv('../data/bowler_match_stats.csv', index=False)
phase_summary.to_csv('../data/phase_summary.csv', index=False)
print("\n Feature engineering complete. Files saved:")
print("  deliveries_featured.csv")
print("  batsman_match_stats.csv")
print("  bowler_match_stats.csv")
print("  phase_summary.csv")

Matches    : (1090, 20)
Deliveries : (260920, 19)
Phase distribution:
phase
Middle       118979
Powerplay     95357
Death         46584
Name: count, dtype: int64

Total boundaries: 42,901
Total dot balls: 90,438
Total wickets  : 12,950

Batsman match stats sample:
   match_id       batter  runs_scored  balls_faced  fours  sixes  strike_rate
0    335982    AA Noffke            9           12      1      0        75.00
1    335982      B Akhil            0            2      0      0         0.00
2    335982  BB McCullum          158           77     10     13       205.19
3    335982     CL White            6           10      0      0        60.00
4    335982    DJ Hussey           12           12      1      0       100.00

Bowler match stats sample:
   match_id      bowler  runs_conceded  legal_balls  wickets_taken  dot_balls  \
0    335982   AA Noffke             41           24              1          6   
1    335982  AB Agarkar             25           24              3         15